In [0]:
%load_ext autoreload
%autoreload 2

# HUD USPS ZIP Code Crosswalk Ingestion

**Purpose:** One-time/infrequent ingestion of the HUD USPS ZIP-to-County crosswalk reference data.

**Run Frequency:** Quarterly (HUD publishes new crosswalk files quarterly)

**Source:** HUD USPS ZIP Code Crosswalk files
- Download page: https://www.huduser.gov/portal/pps_crosswalk.html
- URL pattern: https://www.huduser.gov/portal/datasets/usps/ZIP_COUNTY_YYYYMM.zip
- No API token needed — files are public downloads
- Format: ZIP file containing a CSV with one row per ZIP x county pair

**Output File:**
* `/Volumes/bronze_dev/hud/hud_raw/reference_data/zip_county_crosswalk_YYYYMMDD_HHMMSS.csv`

**Note:** This raw CSV file will be picked up by the DLT bronze pipeline as a batch read (same pattern as states/counties reference data). See GOLD_PLAN.md §1c.

In [0]:
import requests
import zipfile
import io
import os
from datetime import datetime

In [0]:
# ============================================================================
# Configuration
# ============================================================================

# HUD USPS ZIP-county crosswalk download URL.
# Update this to the latest quarter's file before running.
# Find the latest at: https://www.huduser.gov/portal/pps_crosswalk.html
# URL pattern: https://www.huduser.gov/portal/datasets/usps/ZIP_COUNTY_YYYYMM.zip
CROSSWALK_URL = "https://www.huduser.gov/portal/datasets/usps/ZIP_COUNTY_122024.zip"

# Volume path for raw reference data (same directory as states/counties)
VOLUME_PATH = "/Volumes/bronze_dev/hud/hud_raw/reference_data"

In [0]:
# ============================================================================
# Download the ZIP-county crosswalk file from HUD
# ============================================================================

print(f"Downloading from: {CROSSWALK_URL}")
response = requests.get(CROSSWALK_URL)

if response.status_code != 200:
    raise RuntimeError(
        f"Download failed with status {response.status_code}. "
        f"Verify the URL is correct — check https://www.huduser.gov/portal/pps_crosswalk.html "
        f"for the latest crosswalk file."
    )

print(f"Downloaded {len(response.content):,} bytes")

In [0]:
# ============================================================================
# Extract CSV from ZIP and save to volume
# ============================================================================

# Ensure the volume directory exists
os.makedirs(VOLUME_PATH, exist_ok=True)

# Extract the CSV from the ZIP file
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # List contents
    print(f"ZIP contents: {z.namelist()}")
    
    # Find the CSV file inside the ZIP
    csv_files = [f for f in z.namelist() if f.lower().endswith('.csv')]
    
    if not csv_files:
        # Some HUD crosswalk files are Excel (.xlsx) instead of CSV
        xlsx_files = [f for f in z.namelist() if f.lower().endswith('.xlsx')]
        if xlsx_files:
            raise RuntimeError(
                f"Found Excel file ({xlsx_files[0]}) but not CSV. "
                f"HUD may have changed the file format. "
                f"Download and convert to CSV manually, or update this notebook to handle .xlsx."
            )
        raise RuntimeError(f"No CSV file found in ZIP. Contents: {z.namelist()}")
    
    csv_filename = csv_files[0]
    csv_content = z.read(csv_filename)
    print(f"Extracted: {csv_filename} ({len(csv_content):,} bytes)")

# Generate timestamped filename and save
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"zip_county_crosswalk_{timestamp}.csv"
output_path = f"{VOLUME_PATH}/{output_filename}"

with open(output_path, 'wb') as f:
    f.write(csv_content)

print(f"Saved ZIP-county crosswalk to: {output_path}")

In [0]:
# ============================================================================
# Verify: read back the saved CSV and show row count + sample
# ============================================================================

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(output_path)
)

row_count = df.count()
print(f"Total rows: {row_count:,}")
print(f"Columns: {df.columns}")

# Show a few ZIPs that span multiple counties (TOT_RATIO < 1.0)
if "TOT_RATIO" in df.columns:
    multi_county = df.filter(df.TOT_RATIO < 1.0).count()
    print(f"ZIP x county rows where ZIP spans multiple counties: {multi_county:,}")

display(df.limit(10))